In [2]:
import pandas as pd
import torch
from torch_geometric.data import Data
from sklearn.preprocessing import StandardScaler


/home/gowrav8849/myenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:

# =========================
# 2. Load CSVs
# =========================
feat_path = "data/Elliptic_data/elliptic_bitcoin_dataset/elliptic_txs_features.csv"
edge_path = "data/Elliptic_data/elliptic_bitcoin_dataset/elliptic_txs_edgelist.csv"
classes_path = "data/Elliptic_data/elliptic_bitcoin_dataset/elliptic_txs_classes.csv"

features = pd.read_csv(feat_path, header=None)
edges = pd.read_csv(edge_path)
classes = pd.read_csv(classes_path)

# =========================
# 3. Rename columns
# =========================
features.rename(columns={0: "txId", 1: "time_step"}, inplace=True)
classes.rename(columns={"txId": "txId", "class": "label"}, inplace=True)

# =========================
# 4. Merge features + labels
# =========================
df = features.merge(classes, on="txId")

# =========================
# 5. Map labels to integers
# =========================
label_map = {
    "unknown": 0,
    "1": 1,   # illicit
    "2": 2    # licit
}
df["label"] = df["label"].astype(str).map(label_map)

# =========================
# 6. Create node index mapping
# =========================
tx_ids = df["txId"].values
id_map = {tx_id: i for i, tx_id in enumerate(tx_ids)}

# =========================
# 7. Build feature matrix
# =========================
feature_cols = df.columns[2:-1]   # skip txId, time_step, label
x = df[feature_cols].values

# Normalize features
scaler = StandardScaler()
x = scaler.fit_transform(x)

x = torch.tensor(x, dtype=torch.float)

# =========================
# 8. Labels
# =========================
y = torch.tensor(df["label"].values, dtype=torch.long)

# =========================
# 9. Time steps
# =========================
time = torch.tensor(df["time_step"].values)

# =========================
# 10. Build edge_index
# =========================
edge_list = []

for _, row in edges.iterrows():
    src = row["txId1"]
    dst = row["txId2"]

    if src in id_map and dst in id_map:
        edge_list.append([id_map[src], id_map[dst]])

edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()

# =========================
# 11. Create masks (temporal split)
# =========================
# (y != 0) is used to remove the unknown nodes from mask as they dont involve in loss calculation
train_mask = (time <= 34) & (y != 0) # this is for static models -----------------------
val_mask   = (time >= 35) & (time <= 39) & (y != 0) # this is for static models -----------------------
test_mask  = (time >= 40) & (y != 0) # this is for static models -----------------------

# =========================
# 12. Build PyG Data object
# =========================
data = Data(
    x=x,
    edge_index=edge_index,
    y=y,
    train_mask=train_mask,
    val_mask=val_mask,
    test_mask=test_mask
)

print(data)
print("Train nodes:", data.train_mask.sum().item())
print("Val nodes:", data.val_mask.sum().item())
print("Test nodes:", data.test_mask.sum().item())

Data(x=[203769, 165], edge_index=[2, 234355], y=[203769], train_mask=[203769], val_mask=[203769], test_mask=[203769])
Train nodes: 29894
Val nodes: 5486
Test nodes: 11184


**"unknown": 0** **"illicit: 1"** **"licit": 2**

In [8]:
def print_split_stats(data, name, mask):
    y = data.y[mask]

    total = y.size(0)
    num_illicit = (y == 1).sum().item()
    num_licit   = (y == 2).sum().item()

    print(f"\n{name} Split:")
    print(f"Total labeled nodes: {total}")
    print(f"Illicit (1): {num_illicit} ({num_illicit/total:.4f})")
    print(f"Licit   (2): {num_licit} ({num_licit/total:.4f})")

# Run for all splits
print_split_stats(data, "Train", data.train_mask)
print("Unknown in train:", ((data.y == 0) & data.train_mask).sum().item())
print_split_stats(data, "Val", data.val_mask)
print("Unknown in val:",   ((data.y == 0) & data.val_mask).sum().item())
print_split_stats(data, "Test", data.test_mask)
print("Unknown in test:",  ((data.y == 0) & data.test_mask).sum().item())


Train Split:
Total labeled nodes: 29894
Illicit (1): 3462 (0.1158)
Licit   (2): 26432 (0.8842)
Unknown in train: 0

Val Split:
Total labeled nodes: 5486
Illicit (1): 447 (0.0815)
Licit   (2): 5039 (0.9185)
Unknown in val: 0

Test Split:
Total labeled nodes: 11184
Illicit (1): 636 (0.0569)
Licit   (2): 10548 (0.9431)
Unknown in test: 0


In [9]:
print("Unknown in train time (t ≤ 34):",
      ((time <= 34) & (data.y == 0)).sum().item())

print("Unknown in val time (35–39):",
      ((time >= 35) & (time <= 39) & (data.y == 0)).sum().item())

print("Unknown in test time (t ≥ 40):",
      ((time >= 40) & (data.y == 0)).sum().item())

Unknown in train time (t ≤ 34): 106371
Unknown in val time (35–39): 15371
Unknown in test time (t ≥ 40): 35463


In [7]:
from collections import Counter

def detailed_stats(data, mask):
    labels = data.y[mask].tolist()
    print(Counter(labels))

print("\nTrain distribution:", detailed_stats(data, data.train_mask))
print("\nVal distribution:", detailed_stats(data, data.val_mask))
print("\nTest distribution:", detailed_stats(data, data.test_mask))

Counter({2: 26432, 1: 3462})

Train distribution: None
Counter({2: 5039, 1: 447})

Val distribution: None
Counter({2: 10548, 1: 636})

Test distribution: None


In [11]:
neighbors = data.edge_index

unknown_mask = data.y == 0
illicit_mask = data.y == 1

# edges: unknown → illicit
mask1 = (data.y[neighbors[0]] == 0) & (data.y[neighbors[1]] == 1)
mask2 = (data.y[neighbors[0]] == 1) & (data.y[neighbors[1]] == 0)

print("Unknown → Illicit:", mask1.sum().item())
print("Illicit → Unknown:", mask2.sum().item())

Unknown → Illicit: 3993
Illicit → Unknown: 1458


In [12]:
total_unknown_edges = (data.y[neighbors[0]] == 0).sum().item()

print("Ratio:", 3993 / total_unknown_edges)

Ratio: 0.022041654476503806


In [13]:
unknown_neighbors = data.y[neighbors[1]][data.y[neighbors[0]] == 0]

illicit_ratio = (unknown_neighbors == 1).sum().item() / unknown_neighbors.size(0)

print("P(neighbor is illicit | node is unknown):", illicit_ratio)

P(neighbor is illicit | node is unknown): 0.022041654476503806


In [14]:
# illicit → illicit connectivity
mask_illicit = (data.y[neighbors[0]] == 1)
illicit_neighbors = data.y[neighbors[1]][mask_illicit]

print("P(neighbor illicit | illicit node):",
      (illicit_neighbors == 1).sum().item() / illicit_neighbors.size(0))

P(neighbor illicit | illicit node): 0.29605458320973005


In [16]:
#global illicit ratio
global_illicit_ratio = (data.y == 1).sum().item() / (data.y != 0).sum().item()
print(global_illicit_ratio)

0.09760759384932566


**So the graph is locally homophilic (relatively)**

In [ ]:
# as graph 